> This post introduces `helix` - a Claude Code orchestrator that learns from every session.

# Introduction

The criticism is familiar: agents forget everything between sessions.

The word "forget" is load-bearing here. It implies there was something to remember - some mechanism for retention that failed. But agents don't forget. There's no architecture for remembering. Each session starts blank not because memories faded, but because no one built the filing system.

`helix` is that filing system. It's the successor to `ftl`, rebuilt around a single insight: memory without feedback is just storage. A filing cabinet that grows larger is not getting smarter. `helix` tracks which memories actually helped, and adjusts rankings based on that signal.

I've been building with Opus 4.5 since its release, and the transformation from "spastic assistant" to "genuine collaborator" is real. What was missing wasn't model capability - it was architecture that let collaboration compound. `helix` provides that architecture.

# Philosophy

`helix` is built on six principles:

| Principle | What it means |
|-----------|---------------|
| **Feedback closes the loop** | Memories that help rise in ranking. Memories that don't help sink. |
| **Verify first** | Shape work by starting with proof-of-success |
| **Bounded scope** | Delta files are explicit so humans can audit agent boundaries |
| **Present over future** | Implement current requests, not anticipated needs |
| **Edit over create** | Modify what exists before creating something new |
| **Blocking is success** | Clear blocking info is better than broken code |

The word "feedback" in the first principle is load-bearing. In `ftl`, memory accumulated - each task left artifacts that persisted. But persistence is not learning. A filing cabinet that grows larger is not getting smarter.

`helix` closes the loop. When memory is injected into a task, we track whether it actually helped. What happens when a memory is injected but ignored? The system notices. That memory's effectiveness score drops, pushing it down in future rankings. Over time, memories that consistently help rise to the top. Memories that seemed relevant but weren't sink toward oblivion.

I believe this is the critical difference between storage and learning: tracking not just what exists, but what worked.

# The Development Loop

```
/helix <objective>
    │
    ▼
┌─────────────────────────────────────┐
│  EXPLORER (haiku, 6 tools)          │
│  structure │ patterns │ memory │ targets
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  PLANNER (opus)                     │
│  Decompose → Dependencies → Budget  │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  BUILDER (opus, budget 5-9)         │
│  Read → Implement → Verify → Report │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  OBSERVER (opus)                    │
│  Extract failures │ Chunk patterns  │
└─────────────────────────────────────┘
```

Four specialized agents, each with a distinct role. The Explorer gathers context. The Planner decomposes objectives into tasks with dependencies. The Builder executes within strict constraints. The Observer extracts learning from outcomes.

Memory flows through the entire pipeline:

```
recall() → inject → feedback() → store()
```

The Explorer queries memory for relevant context before any planning happens. The Builder receives injected memories and reports which ones actually helped. The Observer stores new failures and patterns. The feedback loop adjusts rankings based on what worked.

What happens when the Builder reports that a memory was never used? The feedback loop records a failure for that memory. Its effectiveness score drops. Next time, that memory ranks lower - or doesn't surface at all. The system is learning which context actually helps, not which context seemed relevant.

Each completed objective makes the system smarter - but only if feedback is honest.

# Agents

At one extreme: a single monolithic agent doing everything - exploration, planning, building, learning. Maximum context, minimum coordination overhead. But also maximum scope creep, maximum token waste when things go sideways.

At the other extreme: dozens of specialized microservices, each responsible for one atomic operation. Minimal blast radius, but coordination overhead dominates. The orchestrator becomes more complex than the work being orchestrated.

`helix` sits between them: four agents, each with distinct capabilities and constraints.

| Agent | Model | Role | Budget |
|-------|-------|------|--------|
| **Explorer** | Haiku | Codebase reconnaissance | 6 |
| **Planner** | Opus | Task DAG decomposition | unlimited |
| **Builder** | Opus | Execution within constraints | 5-9 |
| **Observer** | Opus | Learning extraction | 10 |

The division of labor is deliberate.

**Explorer** runs on Haiku for cost efficiency. Its job is reconnaissance, not reasoning - find the structure, detect the framework, query memory for relevant context, identify target files. Six tool calls is enough for exploration. More would be scope creep.

**Planner** runs on Opus with no tool budget limit. Planning is the highest-leverage activity - a poor plan wastes every downstream tool call. The Planner decomposes complex objectives into focused tasks, sets dependencies (parallelizing where possible), and assigns budgets based on complexity.

**Builder** runs on Opus with a tight budget (5-9 tools per task). This constraint prevents spiral. If a task can't be completed within budget, the Builder blocks with what it tried. The budget forces focus: read the delta files, implement the change, verify, report.

**Observer** runs on Opus with enough budget to analyze outcomes. Its job is extracting generalizable knowledge from what just happened - failures from blocked tasks, patterns from successful completions, relationships between memories.

# Task DAG

The Planner doesn't create a list of tasks. It creates a directed acyclic graph with explicit dependencies.

```
001: spec-auth-models ─┬─→ 002: spec-auth-tests ─┬─→ 004: impl-auth-routes
                       │                         │
                       └─→ 003: impl-auth-service ─┘
```

Each task has:

| Field | Purpose |
|-------|--------|
| `seq` | Execution order identifier ("001", "002") |
| `slug` | Human-readable name ("spec-auth-models") |
| `objective` | What this task accomplishes |
| `delta` | Files this task may modify (strict constraint) |
| `verify` | Command to verify completion |
| `depends` | Tasks that must complete first ("none" or "001,002") |
| `budget` | Tool calls allocated (5-9) |

Tasks are registered with Claude Code's native task system, visible via `Ctrl+T` or `/todos`. This gives humans visibility into progress as the pipeline executes.

What happens when task 002 fails? The DAG structure means 003 can still proceed if it doesn't depend on 002. Failures are contained to their branch. A blocked task doesn't poison the entire objective - only the tasks that explicitly depend on it.

Tasks can run in parallel when they don't depend on each other. In the diagram above, `002` and `003` execute simultaneously once `001` completes. The DAG captures both sequencing constraints and parallelization opportunities.

# Memory System

This is where `helix` diverges most from its predecessor. The memory system isn't storage - it's a learning system with effectiveness tracking and decay.

## Storage

Everything lives in a single SQLite database at `.helix/helix.db`. No scattered JSON files, no complex file hierarchies.

| Table | Purpose |
|-------|--------|
| `memory` | Failures and patterns with embeddings |
| `memory_edge` | Relationships between memories |
| `exploration` | Gathered context from Explorer |
| `plan` | Task decompositions from Planner |
| `workspace` | Task execution contexts |

Memories are stored with 384-dimensional embeddings for semantic search. 384 dimensions. The human mind cannot visualize a 384-dimensional space any more than it can visualize the distance to Andromeda. But similarity in that space captures meaning in ways keyword matching never will. Two memories about "authentication failing silently" and "auth errors swallowed without logging" cluster together, even though they share no keywords.

## Scoring Formula

When retrieving memories, ranking is not just semantic relevance. The scoring formula:

```
score = (0.5 × relevance) + (0.3 × effectiveness) + (0.2 × recency)
```

Where:
- **relevance** = cosine similarity between query and memory embeddings
- **effectiveness** = `helped / (helped + failed)`, default 0.5 if no feedback
- **recency** = `2^(-days_since_use / 7)` (ACT-R decay)

The ACT-R decay comes from cognitive architecture research. Memories that haven't been used recently fade, matching how human memory works. A memory that helped six months ago but hasn't been touched since will have lower recency than one used yesterday.

## The Feedback Loop

This is the critical function:

```python
feedback(utilized, injected)
# utilized memories: helped++
# injected-but-unused: failed++
```

When the Builder completes a task, it reports which memories were actually utilized. The feedback function compares this to what was injected:

- Memory was injected AND utilized? → `helped` counter increments
- Memory was injected but NOT utilized? → `failed` counter increments

Over time, effective memories rise in ranking. Memories that consistently get injected but ignored sink.

I believe this is why most agent memory systems fail: they track what was stored, not what helped. Retrieval without feedback is just search. Retrieval with feedback is learning.

## SOAR Chunking

The Observer extracts patterns using SOAR-style chunking. When a task succeeds with a notable technique:

```bash
python3 $HELIX_PLUGIN_ROOT/lib/memory/core.py chunk \
    --task "What was accomplished" \
    --outcome "SUCCESS" \
    --approach "The technique that worked"
```

This captures the transition from deliberate problem-solving to compiled expertise. A technique that worked once becomes a retrievable pattern for similar future situations.

## Graph Relationships

Memories don't exist in isolation. The system tracks edges between them:

| Type | Meaning |
|------|--------|
| `co_occurs` | These failures tend to appear together |
| `causes` | This failure leads to that failure |
| `solves` | This pattern resolves that failure |
| `similar` | These memories are semantically close |

What happens when a new failure is stored? The `connected()` traversal explores its neighborhood - if this failure appears, what related failures should we watch for? What patterns have solved it before? The graph structure surfaces context that pure embedding similarity would miss.

## Maintenance

The memory system requires occasional maintenance:

| Operation | What it does |
|-----------|-------------|
| `consolidate()` | Merge semantically similar memories |
| `prune()` | Remove memories with effectiveness < 0.25 |
| `decay()` | Find dormant memories that haven't been used |

# Commands

| Command | Purpose |
|---------|--------|
| `/helix <objective>` | Full pipeline: explore → plan → build → observe |
| `/helix-query "topic"` | Search memory by semantic similarity |
| `/helix-stats` | Memory health metrics and feedback loop status |

# Blocking as Learning

The instinct is to keep trying. An agent that gives up feels like failure.

The word "failure" is wrong here. A blocked workspace with clear documentation is information. An agent that spirals for 100k tokens is waste. The confidence to escalate - to say "this is beyond what I can debug, here's what I tried" - is a feature, not a bug.

When a task goes sideways, the Builder has a hard constraint: 5-9 tools max. If it hasn't solved the problem within budget, it's exploring, not debugging. At that point, or after hitting the same error three times, the Builder blocks.

The workspace records what was tried:

```
BLOCKED: Need to modify src/main.py but it's not in delta
TRIED: Implemented auth service in src/services/auth.py
ERROR: Cannot import auth routes without modifying main.py
```

The metacognition check is explicit: after three failed attempts with similar approaches, the Builder must stop and analyze rather than retry. Is there a fundamentally different approach? Is the task mis-scoped? Is information missing?

Blocked workspaces feed the Observer. Every block is a potential failure pattern to extract - a lesson for future tasks encountering similar situations. The system learns from failures, but only if failures are captured cleanly instead of buried in spiral.

# When to Use

**Use helix when:**

- Work should persist as learned knowledge that compounds over time
- Complex objectives need decomposition into verifiable tasks
- Bounded, reviewable scope with explicit file constraints matters
- Past failures should inform future attempts
- Framework-specific development benefits from detected idioms

**Skip helix when:**

- Simple single-file changes don't benefit from orchestration
- Exploratory prototyping where the model should wander freely
- Quick one-offs with no future value

I reach for `helix` when the work will matter again tomorrow. If I'm debugging a one-off script, orchestration overhead isn't worth it. If I'm building infrastructure that will evolve over months, the memory system pays dividends every session.

# Installation

```bash
# Add the crinzo-plugins marketplace
claude plugin marketplace add https://github.com/enzokro/crinzo-plugins

# Install helix
claude plugin install helix@crinzo-plugins
```

Or from inside Claude Code:

```bash
/plugin marketplace add https://github.com/enzokro/crinzo-plugins
/plugin install helix@crinzo-plugins
```

# Conclusion

Context loss is an architecture problem, not a capability problem. The models are ready - Opus 4.5 proved that agents can be genuine collaborators. What was missing was the architecture to let that collaboration compound.

`helix` builds on `ftl` with a crucial addition: feedback. Memory accumulation is not learning. A system that tracks which memories actually helped - and adjusts rankings based on that signal - is a system that improves over time.

Memory without feedback is storage. Memory with feedback is learning.

The models are ready. The architecture is learning. The question is what we'll build when context compounds instead of vanishing.